# TF-IDF + Logistic Regression trên ViGoEmotions (7 nhóm Ekman), seed 42

Baseline cổ điển, cấu hình giống `02_models_baseline.ipynb` của nhóm:
- `TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)` trên cột `text` của `data_ekman7` (S1, giữ emoji, không pyvi; cùng đầu vào với CafeBERT và XLM-R bản nopyvi)
- `OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))`, ngưỡng 0,5 cho mọi nhãn

Kết quả lưu theo format các run transformer (`saved_models/runs_ekman7/<run>/`): `model.joblib` (checkpoint), `val_probs.npy`, `test_probs.npy`, `val_labels.npy`, `test_labels.npy` (float32, cùng thứ tự dòng với `data_ekman7`, ghép ensemble được), `ket_qua.json`, `test_metrics.json`, `classification_report_test.xlsx`, `config.json`, `env.json`.

Đổi `SCHEME = "28"` để chạy 28 nhãn gốc (cột `labels_orig`, cùng câu). Bản script tương đương: `notebooks/train_tfidf_lr.py`.

In [3]:
import ast, json, os, platform, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

# ----- cấu hình -----
SCHEME = "ekman7"      # "ekman7" (7 nhóm) hoặc "28" (28 nhãn gốc)
SEED = 42
THRESHOLD = 0.5
TEXT_COL = "text"      # "text_raw" = chữ gốc chưa làm sạch
DATA_DIR = None        # None: tự tìm data_ekman7; hoặc ghi đường dẫn, ví dụ r"C:\Users\ten\Downloads\data_ekman7"

np.random.seed(SEED)
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCHEMES = {"ekman7": {"col": "labels", "out": "runs_ekman7", "tag": "ekman7"},
           "28": {"col": "labels_orig", "out": "runs_28nhan", "tag": "28nhan"}}
print("sklearn", sklearn.__version__, "| project:", PROJECT_DIR.name)

sklearn 1.8.0 | project: DAP391m-sentiment-vn


In [4]:
def find_data_dir():
    cands = [DATA_DIR, os.getenv("DATA_EKMAN7"), PROJECT_DIR / "data" / "data_ekman7",
             PROJECT_DIR.parent / "data_ekman7", Path.home() / "Downloads" / "data_ekman7"]
    for d in cands:
        if d and (Path(d) / "train.csv").exists() and (Path(d) / "val.csv").exists():
            return Path(d)
    raise FileNotFoundError("Không tìm thấy data_ekman7 (train.csv, val.csv, test.csv). Điền DATA_DIR ở ô trên.")

data_dir = find_data_dir()
if SCHEME == "ekman7":
    d = json.loads((data_dir / "label_dict_ekman7.json").read_text(encoding="utf-8"))
else:
    d = json.loads((PROJECT_DIR / "data" / "raw" / "vigoemotions" / "label_dict.json").read_text(encoding="utf-8"))
NAMES = [d[str(i)] for i in range(len(d))]
K = len(NAMES)

def to_onehot(series, k):
    y = np.zeros((len(series), k), dtype=np.float32)
    for i, v in enumerate(series):
        for j in (ast.literal_eval(v) if isinstance(v, str) else v):
            y[i, int(j)] = 1.0
    return y

read = lambda s: pd.read_csv(data_dir / f"{s}.csv", encoding="utf-8-sig", keep_default_na=False, dtype=str)
tr, va, te = read("train"), read("val"), read("test")
col = SCHEMES[SCHEME]["col"]
y_tr, y_va, y_te = to_onehot(tr[col], K), to_onehot(va[col], K), to_onehot(te[col], K)
print(f"{data_dir.name}: train {len(tr)}, val {len(va)}, test {len(te)} | {K} nhãn: {NAMES if K <= 7 else str(K) + ' nhãn gốc'}")

data_ekman7_neutral_wins: train 16531, val 2066, test 2067 | 7 nhãn: ['joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust', 'neutral']


In [5]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
    ("clf", OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))),
])
t0 = time.perf_counter()
model.fit(tr[TEXT_COL].tolist(), y_tr.astype(int))
fit_s = time.perf_counter() - t0
n_feat = len(model.named_steps["tfidf"].vocabulary_)
print(f"Huấn luyện xong sau {fit_s:.1f} giây, {n_feat:,} đặc trưng TF-IDF")

Huấn luyện xong sau 1.3 giây, 33,369 đặc trưng TF-IDF


In [6]:
def metrics(y, p, thr=THRESHOLD):
    pred = (p >= thr).astype(int)
    f = lambda avg: round(100 * f1_score(y, pred, average=avg, zero_division=0), 2)
    return {"macro": f("macro"), "weighted": f("weighted"), "micro": f("micro"), "samples": f("samples"),
            "exact_match": round(100 * float((pred == y).all(axis=1).mean()), 2)}

p_va = model.predict_proba(va[TEXT_COL].tolist()).astype(np.float32)
p_te = model.predict_proba(te[TEXT_COL].tolist()).astype(np.float32)
m_va, m_te = metrics(y_va, p_va), metrics(y_te, p_te)
print(pd.DataFrame({"val": m_va, "test": m_te}).T)
report = pd.DataFrame(classification_report(y_te, (p_te >= THRESHOLD).astype(int), target_names=NAMES, digits=4,
                                            zero_division=0, output_dict=True)).T
report.round(4)

      macro  weighted  micro  samples  exact_match
val   56.32     64.90  63.69    64.91        39.11
test  55.55     65.23  64.09    65.14        36.96


,precision,recall,f1-score,support
joy,0.7754,0.7488,0.7619,1019.0
sadness,0.6178,0.7500,0.6775,528.0
anger,0.5856,0.7691,0.6649,485.0
fear,0.4925,0.6490,0.5600,151.0
surprise,0.4432,0.6161,0.5155,310.0
disgust,0.3295,0.6099,0.4279,141.0
neutral,0.2330,0.3534,0.2808,116.0
micro avg,0.5852,0.7084,0.6409,2750.0
macro avg,0.4967,0.6423,0.5555,2750.0
weighted avg,0.6129,0.7084,0.6523,2750.0


In [7]:
# ── 5-fold Cross-Validation — TF-IDF + LR (training set) ─────────────────────
# DL models (CafeBERT/PhoBERT/ViSoBERT/XLMR): không thực hiện k-fold CV vì mỗi fold
# mất nhiều giờ GPU — thực hành chuẩn trong NLP papers (EACL, EMNLP).
# TF-IDF (~2s/fold): thực hiện 5-fold để kiểm tra độ ổn định trên training set.

from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
X_tr_list = tr[TEXT_COL].tolist()
cv_macro, cv_weighted = [], []

print(f"5-fold Cross-Validation — TF-IDF + LR  (seed={SEED}, threshold={THRESHOLD})")
print("-" * 56)
for fold, (ti, vi) in enumerate(kf.split(X_tr_list), 1):
    cv_m = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        ("clf",   OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced",
                                                          random_state=SEED))),
    ])
    X_f_tr = [X_tr_list[i] for i in ti]
    X_f_va = [X_tr_list[i] for i in vi]
    y_f_tr, y_f_va = y_tr[ti], y_tr[vi]
    cv_m.fit(X_f_tr, y_f_tr.astype(int))
    pred_fold = (cv_m.predict_proba(X_f_va) >= THRESHOLD).astype(int)
    mac = f1_score(y_f_va, pred_fold, average="macro",    zero_division=0)
    wgt = f1_score(y_f_va, pred_fold, average="weighted", zero_division=0)
    cv_macro.append(mac); cv_weighted.append(wgt)
    print(f"Fold {fold}:  F1-macro={mac*100:.2f}%   F1-weighted={wgt*100:.2f}%")

print("-" * 56)
print(f"Mean  F1-macro:    {np.mean(cv_macro)*100:.2f}% ± {np.std(cv_macro)*100:.2f}%")
print(f"Mean  F1-weighted: {np.mean(cv_weighted)*100:.2f}% ± {np.std(cv_weighted)*100:.2f}%")
print(f"Test  F1-macro:    {m_te['macro']:.2f}%  (hold-out — consistent với CV)")

5-fold Cross-Validation — TF-IDF + LR  (seed=42, threshold=0.5)
--------------------------------------------------------
Fold 1:  F1-macro=53.26%   F1-weighted=63.07%
Fold 2:  F1-macro=54.97%   F1-weighted=64.75%
Fold 3:  F1-macro=54.36%   F1-weighted=64.63%
Fold 4:  F1-macro=55.47%   F1-weighted=64.46%
Fold 5:  F1-macro=54.88%   F1-weighted=63.90%
--------------------------------------------------------
Mean  F1-macro:    54.59% ± 0.75%
Mean  F1-weighted: 64.16% ± 0.62%
Test  F1-macro:    55.55%  (hold-out — consistent với CV)


In [8]:
# ── Bảng so sánh 5 mô hình — test set ViGoEmotions Ekman-7 ───────────────────
# Đọc classification_report_test.xlsx từ từng thư mục runs/

paper_rows = [
    {"Mô hình": "ViSoBERT 28-label (EACL 2026, paper baseline)",
     "Loại": "Paper baseline", "F1-macro": 61.50, "F1-weighted": 63.26, "Nhãn": 28},
]
tfidf_row = {
    "Mô hình": "TF-IDF + LR (Ekman-7)", "Loại": "Classical (ours)",
    "F1-macro": m_te["macro"], "F1-weighted": m_te["weighted"], "Nhãn": 7,
}
dl_map = {
    "CafeBERT": "Vietnamese DL — uitnlp/CafeBERT",
    "PhoBERT":  "Vietnamese DL — vinai/phobert-base-v2",
    "ViSoBERT": "Vietnamese DL — uitnlp/visobert",
    "XLMR":     "Multilingual DL — xlm-roberta-base",
}
rows = paper_rows + [tfidf_row]
for mname, mtype in dl_map.items():
    rpt_path = PROJECT_DIR / "runs" / mname / "classification_report_test.xlsx"
    if rpt_path.exists():
        rpt = pd.read_excel(rpt_path, index_col=0)
        rows.append({
            "Mô hình": mname, "Loại": mtype,
            "F1-macro":    round(rpt.loc["macro avg",    "f1-score"] * 100, 2),
            "F1-weighted": round(rpt.loc["weighted avg", "f1-score"] * 100, 2),
            "Nhãn": 7,
        })
    else:
        print(f"WARNING: {rpt_path} not found")

df_cmp = (pd.DataFrame(rows)
            .sort_values("F1-macro", ascending=False)
            .reset_index(drop=True))
df_cmp.index += 1
df_show = df_cmp.copy()
df_show["F1-macro"]    = df_show["F1-macro"].apply(lambda x: f"{x:.2f}%")
df_show["F1-weighted"] = df_show["F1-weighted"].apply(lambda x: f"{x:.2f}%")

print("=" * 76)
print("  So sánh 5 mô hình — test set ViGoEmotions Ekman-7 (2,067 samples)")
print("=" * 76)
print(df_show[["Mô hình", "Loại", "Nhãn", "F1-macro", "F1-weighted"]].to_string())
print("=" * 76)
best = df_cmp.iloc[0]
paper_macro = 61.50
print(f"→ {best['Mô hình']} dẫn đầu F1-macro ({best['F1-macro']:.2f}%), "
      f"vượt paper baseline +{best['F1-macro']-paper_macro:.1f}pp")
print(f"→ TF-IDF baseline thấp hơn best DL {best['F1-macro']-m_te['macro']:.1f}pp"
      f" — DL học được đặc trưng ngữ nghĩa sâu hơn n-gram")

  So sánh 5 mô hình — test set ViGoEmotions Ekman-7 (2,067 samples)
                                         Mô hình                                   Loại  Nhãn F1-macro F1-weighted
1                                       CafeBERT        Vietnamese DL — uitnlp/CafeBERT     7   67.20%      75.97%
2                                        PhoBERT  Vietnamese DL — vinai/phobert-base-v2     7   64.85%      73.77%
3                                       ViSoBERT        Vietnamese DL — uitnlp/visobert     7   64.66%      74.19%
4                                           XLMR     Multilingual DL — xlm-roberta-base     7   62.81%      72.94%
5  ViSoBERT 28-label (EACL 2026, paper baseline)                         Paper baseline    28   61.50%      63.26%
6                          TF-IDF + LR (Ekman-7)                       Classical (ours)     7   55.55%      65.23%
→ CafeBERT dẫn đầu F1-macro (67.20%), vượt paper baseline +5.7pp
→ TF-IDF baseline thấp hơn best DL 11.7pp — DL học được đặc tr

In [9]:
RUN = f"tfidf_lr_{SCHEMES[SCHEME]['tag']}_seed{SEED}"
out = PROJECT_DIR / "saved_models" / SCHEMES[SCHEME]["out"] / RUN
out.mkdir(parents=True, exist_ok=True)

joblib.dump(model, out / "model.joblib", compress=3)                     # checkpoint
for name, arr in [("val_probs", p_va), ("test_probs", p_te), ("val_labels", y_va), ("test_labels", y_te)]:
    np.save(out / f"{name}.npy", arr)
cfg = {"run": RUN, "model_name": "TF-IDF + Logistic Regression", "seed": SEED, "threshold": THRESHOLD,
       "label_scheme": SCHEME, "label_names": NAMES, "data_tag": data_dir.name, "text_col": TEXT_COL,
       "tfidf": {"ngram_range": [1, 2], "min_df": 2, "max_df": 0.95, "sublinear_tf": True},
       "classifier": {"type": "OneVsRestClassifier(LogisticRegression)", "max_iter": 1000,
                      "class_weight": "balanced", "solver": "lbfgs", "C": 1.0, "random_state": SEED},
       "note": "Cấu hình giống notebooks/02_models_baseline.ipynb của nhóm."}
(out / "config.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding="utf-8")
(out / "ket_qua.json").write_text(json.dumps({"run": RUN, "model": cfg["model_name"], "label_scheme": SCHEME,
    "threshold": THRESHOLD, "val": m_va, "test": m_te, "n_features": n_feat, "fit_seconds": round(fit_s, 1)},
    ensure_ascii=False, indent=1), encoding="utf-8")
(out / "test_metrics.json").write_text(json.dumps({"macro_f1": m_te["macro"], "weighted_f1": m_te["weighted"],
    "micro_f1": m_te["micro"], "samples_f1": m_te["samples"], "exact_match": m_te["exact_match"],
    "val_macro_f1": m_va["macro"]}, indent=1), encoding="utf-8")
report.to_excel(out / "classification_report_test.xlsx")
(out / "env.json").write_text(json.dumps({"python": platform.python_version(), "scikit-learn": sklearn.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__,
    "platform": platform.platform()}, indent=1), encoding="utf-8")
(out / "DONE").write_text("ok", encoding="utf-8")
print("Đã lưu vào", out.relative_to(PROJECT_DIR))
print(sorted(p.name for p in out.iterdir()))

Đã lưu vào saved_models\runs_ekman7\tfidf_lr_ekman7_seed42
['DONE', 'classification_report_test.xlsx', 'config.json', 'env.json', 'ket_qua.json', 'model.joblib', 'test_labels.npy', 'test_metrics.json', 'test_probs.npy', 'val_labels.npy', 'val_probs.npy']


## Nạp lại checkpoint và dự đoán

In [10]:
m = joblib.load(out / "model.joblib")
assert np.allclose(m.predict_proba(te[TEXT_COL].tolist()), p_te, atol=1e-6)   # nạp lại cho đúng xác suất đã lưu
for s in ["hóng mãi cuối cùng bị cho leo cây 😒", "cảm ơn thầy nhiều ạ"]:
    p = m.predict_proba([s])[0]
    print(s, "->", {NAMES[j]: round(float(p[j]), 2) for j in np.argsort(-p)[:3]})

hóng mãi cuối cùng bị cho leo cây 😒 -> {'surprise': 0.5, 'joy': 0.47, 'sadness': 0.42}
cảm ơn thầy nhiều ạ -> {'joy': 0.98, 'neutral': 0.1, 'surprise': 0.09}
